# Fig. 9: unfolding depth

Run cells in order from a fresh kernel. Settings are in `configs/experiments.json`; artifacts use the paths described in the README. Timings depend on hardware and software.


In [ ]:
from fthbf.runtime import (DATA_DIR, CHECKPOINT_DIR, checkpoint_path, load_artifact,
                           result_dir, read_config, seed_everything, record_run)
import torch
cfg = read_config("experiments.json")["Fig04"]
seed_everything(cfg["seed"])
record_run("Fig04", cfg)


In [ ]:
with torch.no_grad():
    # =====================================================
    # Evaluate WSR & Runtime vs I_max for Unfolded WMMSE
    # Plot mean + min–max range
    # =====================================================

    import torch
    import numpy as np
    import matplotlib.pyplot as plt
    from tqdm import tqdm
    import time
    import os

    from solutions.trihybrid_WMMSE import TriHybridWMMSE, GNN
    from solutions.utils import computeWSR, channelDataset, HybridPrecodingBFGS, checkHybridConstraints


    # =====================================================
    # Config
    # =====================================================
    Imax_list = cfg["unfolded_layers"]
    Nrf = cfg["rf_chains"]
    RESULT_DIR = result_dir("results_WSRRuntime_vs_Imax")
    os.makedirs(RESULT_DIR, exist_ok=True)
    cdtype = torch.cfloat


    # =====================================================
    # Load problem instances
    # =====================================================
    SAVE_DIR = DATA_DIR
    data = load_artifact(os.path.join(SAVE_DIR, "probInsts_test.pt"))
    parameters = load_artifact(os.path.join(SAVE_DIR, "parameters.pt"))

    ProbInsts = data["ProbInsts"]

    Nt = parameters["Nt"]
    Nc = parameters["Nc"]
    P = parameters["P"]
    Mk = parameters["Mk"]
    eta = parameters["eta"]
    Ns_k = parameters["Ns_k"]
    sigma2 = parameters["sigma2"]


    # =====================================================
    # Dataset
    # =====================================================
    dataset = channelDataset(ProbInsts, parameters)
    in_dim = 2 * ProbInsts[0]["H"][0].numel() + 1


    # =====================================================
    # MAIN LOOP over I_max
    # =====================================================
    for I_max in Imax_list:

        print("\n======================================")
        print(f"Evaluating I_max = {I_max}")
        print("======================================")

        # ---------- build model ----------
        GNNs_gamma = torch.nn.ModuleList()
        GNNs_u = torch.nn.ModuleList()

        for _ in range(I_max):
            GNNs_gamma.append(
                GNN(in_dim=in_dim,
                    hidden_dim=2*Ns_k**2+4,
                    out_dim=2*Ns_k**2,
                    num_layers=3,
                    dp_rate=0.1)
            )
            GNNs_u.append(
                GNN(in_dim=in_dim,
                    hidden_dim=Nt+10,
                    out_dim=Nt,
                    num_layers=3,
                    dp_rate=0.1)
            )

        # ---------- load checkpoint ----------
        ckpt_path = checkpoint_path(I_max)
        checkpoint = load_artifact(ckpt_path)

        for i in range(I_max):
            GNNs_gamma[i].load_state_dict(checkpoint["GNNs_gamma_state_dict"][i])
            GNNs_u[i].load_state_dict(checkpoint["GNNs_u_state_dict"][i])

        GNNs_gamma.eval()
        GNNs_u.eval()
        print("[Info] Model loaded")

        # ---------- run dataset ----------
        sum_rates = []
        runtimes = []

        for pro_ind in tqdm(range(min(cfg["samples"], len(dataset))), desc=f"I_max={I_max}"):

            pro_inst = dataset.__getitem__(pro_ind)
            H_list = pro_inst["H_list"]
            K = pro_inst["K"]

            Ns = Ns_k * K

            u0 = [torch.ones(Nc, 1, dtype=cdtype)] * Nt
            v0 = [torch.ones(Ns, 1, dtype=cdtype)] * Nt

            for n in range(Nt):
                norm_vn = torch.linalg.norm(v0[n])
                if norm_vn > torch.sqrt(P):
                    v0[n] = v0[n] / norm_vn * torch.sqrt(P)

            node_features = pro_inst["node_feature"]
            edge_index = pro_inst["edge_index"]

            delta_gamma_list = []
            delta_u_list = []

            graph_id = torch.zeros(K, dtype=torch.int32)
            num_graphs = torch.tensor(1, dtype=torch.int32)

            for i in range(I_max):
                delta_gamma = GNNs_gamma[i](node_features, edge_index)
                delta_gamma_list.append(delta_gamma)

                delta_u = GNNs_u[i](node_features, edge_index, graph_id, num_graphs)
                delta_u_list.append(delta_u.reshape(Nt, 1))

            # ---------- timing ----------
            t0 = time.perf_counter()

            solver = TriHybridWMMSE(
                H_list=H_list,
                Ns_k=[Ns_k]*K,
                Mk=[Mk]*K,
                sigma2=[sigma2]*K,
                beta=[1.0]*K,
                P=[P]*Nt,
                eta=eta,
                u_init=u0,
                v_init=v0
            )

            _, FA_opt, FD_opt, _, _ = solver.run_unfolded(
                dgamma_list=delta_gamma_list,
                du_list=delta_u_list,
                I_max=I_max,
                verbose=False
            )

            FRF, FBB = HybridPrecodingBFGS(FD_opt, Nrf).solve(
                max_iter=100,
                use_B0=True,
                verbose=False
            )

            _, papc_ok, info = checkHybridConstraints(FRF, FBB, P, tol=1e-6, verbose=False)
            if not papc_ok:
                FBB = FBB / torch.sqrt(torch.tensor(info["max_papc_ratio"]))

            t1 = time.perf_counter()

            R_final, _ = computeWSR(FA_opt, FRF, FBB, H_list, parameters)

            sum_rates.append(R_final.item())
            runtimes.append(t1 - t0)


        # ---------- statistics ----------
        sum_rates = np.array(sum_rates)
        runtimes = np.array(runtimes)

        stats = {
            "mean_wsr": sum_rates.mean(),
            "min_wsr": sum_rates.min(),
            "max_wsr": sum_rates.max(),
            "mean_rt": runtimes.mean(),
            "min_rt": runtimes.min(),
            "max_rt": runtimes.max(),
        }

        print(stats)

        np.savez(
            f"{RESULT_DIR}/stats_Imax_{I_max}.npz",
            sum_rates=sum_rates,
            runtimes=runtimes,
            **stats
        )

    print("\n===== Evaluation finished =====")

In [ ]:
# =====================================================
# LOAD results and plot
# =====================================================
print("Loading results and plotting...")

mean_wsr, min_wsr, max_wsr = [], [], []
mean_rt, min_rt, max_rt = [], [], []

for L in Imax_list:
    data = np.load(f"{RESULT_DIR}/stats_Imax_{L}.npz")

    mean_wsr.append(data["mean_wsr"])
    min_wsr.append(data["min_wsr"])
    max_wsr.append(data["max_wsr"])

    mean_rt.append(data["mean_rt"])
    min_rt.append(data["min_rt"])
    max_rt.append(data["max_rt"])


x = np.arange(len(Imax_list))
cap = 0.15

print("\nDone.")

# =====================================================
# Plot WSR range
# =====================================================
plt.figure(figsize=(7,4))

plt.vlines(x, min_wsr, max_wsr, color="black", linewidth=2)
plt.hlines(min_wsr, x-cap, x+cap, color="black", linewidth=2)
plt.hlines(max_wsr, x-cap, x+cap, color="black", linewidth=2)

plt.plot(x, mean_wsr, color="red", linewidth=2)
plt.scatter(x, mean_wsr, color="red", s=70, label="Mean")

plt.xticks(x, [f"L={L}" for L in Imax_list])
plt.xlabel("Number of Unfolding Layers")
plt.ylabel("Optimized WSR (bps/Hz)")
plt.title("WSR Mean and Range vs Unfolding Depth")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================
# Print TikZ table for PGFPlots
# =====================================================
print("\nTikZ table (copy into LaTeX):\n")
print("x mean min max \\\\")

for i, L in enumerate(Imax_list):
    print(
        f"{L} "
        f"{mean_wsr[i]:.6f} "
        f"{min_wsr[i]:.6f} "
        f"{max_wsr[i]:.6f} \\\\"
    )